In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

## Data Preparation

In [3]:
# pip install curl_cffi

In [4]:
#pip install -U yfinance

In [5]:
from curl_cffi import requests as cf_requests
import yfinance as yf

session = cf_requests.Session(impersonate="chrome")


tickers = ["BMW.DE", "MBG.DE"]

prices = yf.download(
    tickers,
    start="2021-01-01",
    auto_adjust=True,
    progress=False,
    session=session
)["Close"]

prices.head()

Ticker,BMW.DE,MBG.DE
Date,,
2021-01-04,50.774330,31.665838
2021-01-05,50.067753,31.660271
2021-01-06,50.159603,31.404270
2021-01-07,50.180801,32.161140
2021-01-08,49.806316,32.105488


In [6]:
prices = prices.rename(columns={"BMW.DE": "BMW", "MBG.DE": "MBG"})

In [7]:
print(prices.columns, prices.size, prices.shape)  

Index(['BMW', 'MBG'], dtype='object', name='Ticker') 2852 (1426, 2)


In [8]:
prices.to_csv("bmw_mbg_prices.csv")

# later, to load it back:
prices = pd.read_csv("bmw_mbg_prices.csv", index_col="Date", parse_dates=True)

## Data Loading / Analysis

In [9]:
prices = pd.read_csv("bmw_mbg_prices.csv", index_col="Date", parse_dates=True)
prices.head()

,BMW,MBG
Date,,
2021-01-04,50.774330,31.665838
2021-01-05,50.067753,31.660271
2021-01-06,50.159603,31.404270
2021-01-07,50.180801,32.161140
2021-01-08,49.806316,32.105488


In [10]:
# Create ratio DataFrame
ratio_df = pd.DataFrame(index=prices.index)
ratio_df["BMW"] = prices["BMW"]
ratio_df["MBG"] = prices["MBG"]
ratio_df["Ratio"] = prices["BMW"] / prices["MBG"]

ratio_df.tail()

,BMW,MBG,Ratio
Date,,,
2026-07-31,59.299999,46.730000,1.268992
2026-08-03,60.060001,47.950001,1.252555
2026-08-04,59.959999,48.209999,1.243725
2026-08-05,59.099998,47.259998,1.250529
2026-08-06,58.840000,46.994999,1.252048


In [11]:
ratio_df["Ratio_Return"] = ratio_df["Ratio"].pct_change()

In [12]:
ratio_df.dropna(inplace=True)
ratio_df.tail()

,BMW,MBG,Ratio,Ratio_Return
Date,,,,
2026-07-31,59.299999,46.730000,1.268992,-0.006821
2026-08-03,60.060001,47.950001,1.252555,-0.012953
2026-08-04,59.959999,48.209999,1.243725,-0.007049
2026-08-05,59.099998,47.259998,1.250529,0.005470
2026-08-06,58.840000,46.994999,1.252048,0.001215


In [13]:
levels = [2.4,2.2,2,1.8,1.6,1.5,1.4,1.3,1.2,1]
sorted_levels = np.sort(np.array(levels))


print("list:",levels,"\nsorted array:",sorted_levels)


list: [2.4, 2.2, 2, 1.8, 1.6, 1.5, 1.4, 1.3, 1.2, 1] 
sorted array: [1.  1.2 1.3 1.4 1.5 1.6 1.8 2.  2.2 2.4]


# **REVERSION SIGNAL GENERATION**

## **1. FUNCTION TO FIND THE SUPPORT AND RESISTANCE FOR EACH ROW**

In [14]:
def find_support_resistance(ratio_series, levels):

    supports = []
    resistances = []

    for price in ratio_series:
        below = levels[levels < price]
        above = levels[levels > price]

        support = below.max() if len(below) > 0 else np.nan
        resistance = above.min() if len(above) > 0 else np.nan

        supports.append(support)
        resistances.append(resistance)

    result = pd.DataFrame({
        "Ratio": ratio_series,
        "Support": supports,
        "Resistance": resistances
    })

    return result

In [15]:
# Checking for 1 row example ...

above = sorted_levels[sorted_levels>1.4]
below = sorted_levels[sorted_levels<1.4]

support = below.max() if len(below)>0 else np.nan
resistance = above.min() if len(above)>0 else np.nan

print("Checking...","\nlevels above: ", above,"\nlevels below: ", below)
print("")
print("resistance: ",resistance)
print("support: ",support)


Checking... 
levels above:  [1.5 1.6 1.8 2.  2.2 2.4] 
levels below:  [1.  1.2 1.3]

resistance:  1.5
support:  1.3


In [16]:
type(ratio_df["Ratio_Return"])

pandas.core.series.Series

In [17]:
df = find_support_resistance(ratio_df["Ratio"], sorted_levels)

In [18]:
df.tail(20)

,Ratio,Support,Resistance
Date,,,
2026-07-10,1.321664,1.3,1.4
2026-07-13,1.307744,1.3,1.4
2026-07-14,1.270672,1.2,1.3
2026-07-15,1.272983,1.2,1.3
2026-07-16,1.271696,1.2,1.3
2026-07-17,1.286311,1.2,1.3
2026-07-20,1.292567,1.2,1.3
2026-07-21,1.281817,1.2,1.3
2026-07-22,1.285619,1.2,1.3


## **2. FUNCTION TO GET THE NORMALIZED PX**

In [19]:
def add_normalized_price(df):
    df = df.copy()
    df["norm_price"] = (df["Ratio"] - df["Support"]) / (df["Resistance"] - df["Support"])
    return df

In [20]:
df = add_normalized_price(df)

In [21]:
df.tail()

,Ratio,Support,Resistance,norm_price
Date,,,,
2026-07-31,1.268992,1.2,1.3,0.689921
2026-08-03,1.252555,1.2,1.3,0.525548
2026-08-04,1.243725,1.2,1.3,0.437254
2026-08-05,1.250529,1.2,1.3,0.505290
2026-08-06,1.252048,1.2,1.3,0.520481


## **3. FUNCTION TO GET THE BOUNCE ZONE**

In [22]:
def get_zone(norm):
    if 0.10 <= norm <= 0.35:
        return "rev sup bounce"
    elif 0.65 <= norm <= 0.90:
        return "rev res bounce"
    else:
        return None

In [23]:
def add_bounce_zone(df):
    df = df.copy()
    df["bounce_zone"] = df["norm_price"].apply(get_zone)
    return df

In [24]:
df = add_bounce_zone(df)

In [25]:
df.tail(20)

,Ratio,Support,Resistance,norm_price,bounce_zone
Date,,,,,
2026-07-10,1.321664,1.3,1.4,0.216640,rev sup bounce
2026-07-13,1.307744,1.3,1.4,0.077443,None
2026-07-14,1.270672,1.2,1.3,0.706717,rev res bounce
2026-07-15,1.272983,1.2,1.3,0.729826,rev res bounce
2026-07-16,1.271696,1.2,1.3,0.716964,rev res bounce
2026-07-17,1.286311,1.2,1.3,0.863108,rev res bounce
2026-07-20,1.292567,1.2,1.3,0.925668,None
2026-07-21,1.281817,1.2,1.3,0.818172,rev res bounce
2026-07-22,1.285619,1.2,1.3,0.856188,rev res bounce


## **4. FUNCTION FOR TOUCH CHECK**

In [26]:
df.shape

(1425, 5)

In [27]:
len(df)

1425

In [28]:
zone = df["bounce_zone"].iloc[0]
zone

'rev res bounce'

In [29]:
for i in range(len(df) - 1410):
    print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14


In [30]:
for days_back in range(5,0,-1):
    price = df["Ratio"].iloc[6 - days_back]
    print(price)

print("\n\nchecking..")
print(df["Ratio"].iloc[1])
print(df["Ratio"].iloc[6-5])
#print(df["Ratio"].iloc[6])
print(df["Ratio"].iloc[5])
print(df["Ratio"].iloc[6-1])

1.5972223791218179
1.560292971645801
1.5513334246644679
1.539555080272142
1.5253943451828211


checking..
1.5972223791218179
1.5972223791218179
1.5253943451828211
1.5253943451828211


In [31]:
def add_touch_check(df):
    df = df.copy()
    
    touch_days_ago = []

    for i in range(len(df)):
        zone = df["bounce_zone"].iloc[i]
        support = df["Support"].iloc[i]
        resistance = df["Resistance"].iloc[i]

        touch = None # default value

        if zone == "rev res bounce":
            for days_back in range(5,0,-1):
                price = df["Ratio"].iloc[i - days_back]
                if abs(price - resistance) / resistance <= 0.005:
                    touch = days_back
                    break

        elif zone == "rev sup bounce":
            for days_back in [5, 4, 3, 2, 1]:
                price = df["Ratio"].iloc[i - days_back]
                if abs(price - support) / support <= 0.005:
                    touch = days_back
                    break

        touch_days_ago.append(touch)

    df["touch_days_ago"] = touch_days_ago
    return df

In [32]:
df = add_touch_check(df)

## **5. FUNCTION FOR 15D AVG PX BEFORE TOUCH**

In [33]:
df[1400:1407]

,Ratio,Support,Resistance,norm_price,bounce_zone,touch_days_ago
Date,,,,,,
2026-07-03,1.338638,1.3,1.4,0.386384,None,NaN
2026-07-06,1.324083,1.3,1.4,0.240833,rev sup bounce,4.0
2026-07-07,1.321296,1.3,1.4,0.212965,rev sup bounce,5.0
2026-07-08,1.328045,1.3,1.4,0.280453,rev sup bounce,NaN
2026-07-09,1.327563,1.3,1.4,0.275626,rev sup bounce,NaN
2026-07-10,1.321664,1.3,1.4,0.216640,rev sup bounce,NaN
2026-07-13,1.307744,1.3,1.4,0.077443,None,NaN


In [34]:
df[1401:1402]

,Ratio,Support,Resistance,norm_price,bounce_zone,touch_days_ago
Date,,,,,,
2026-07-06,1.324083,1.3,1.4,0.240833,rev sup bounce,4.0


In [35]:
df[1401-4:1402]

,Ratio,Support,Resistance,norm_price,bounce_zone,touch_days_ago
Date,,,,,,
2026-06-30,1.303734,1.3,1.4,0.037341,None,NaN
2026-07-01,1.329071,1.3,1.4,0.290706,rev sup bounce,1.0
2026-07-02,1.343001,1.3,1.4,0.430006,None,NaN
2026-07-03,1.338638,1.3,1.4,0.386384,None,NaN
2026-07-06,1.324083,1.3,1.4,0.240833,rev sup bounce,4.0


In [36]:
df.iloc[0]

Ratio                   1.581406
Support                      1.5
Resistance                   1.6
norm_price              0.814063
bounce_zone       rev res bounce
touch_days_ago               NaN
Name: 2021-01-05 00:00:00, dtype: object

In [37]:
df[0:1]

,Ratio,Support,Resistance,norm_price,bounce_zone,touch_days_ago
Date,,,,,,
2021-01-05,1.581406,1.5,1.6,0.814063,rev res bounce,NaN


In [38]:
touch_index = 1401-4
df[touch_index-15: touch_index]

,Ratio,Support,Resistance,norm_price,bounce_zone,touch_days_ago
Date,,,,,,
2026-06-09,1.440520,1.4,1.5,0.405198,None,NaN
2026-06-10,1.435364,1.4,1.5,0.353641,None,NaN
2026-06-11,1.402762,1.4,1.5,0.027617,None,NaN
2026-06-12,1.395397,1.3,1.4,0.953973,None,NaN
2026-06-15,1.384101,1.3,1.4,0.841012,rev res bounce,2.0
2026-06-16,1.389116,1.3,1.4,0.891162,rev res bounce,3.0
2026-06-17,1.331337,1.3,1.4,0.313369,rev sup bounce,NaN
2026-06-18,1.339462,1.3,1.4,0.394620,None,NaN
2026-06-19,1.324793,1.3,1.4,0.247929,rev sup bounce,NaN


In [39]:
print(df[touch_index-15: touch_index].shape)

(15, 6)


In [40]:
def add_avg_15d_at_touch(df):
    df = df.copy()
    
    avg_15d = []
    
    for i in range(len(df)):
        touch = df["touch_days_ago"].iloc[i]
        
        if pd.isna(touch):
            avg_15d.append(None)
            continue
        
        touch = int(touch)  # in case it got stored as float (e.g. 5.0)
        touch_index = i - touch
        
        if touch_index - 15 < 0:
            avg_15d.append(None)
            continue
        
        window = df["Ratio"].iloc[touch_index - 15 : touch_index]
        avg_15d.append(window.mean())
    
    df["avg_15d_at_touch"] = avg_15d
    return df

In [41]:
df = add_avg_15d_at_touch(df)

In [42]:
df[1402-15 : 1405]

,Ratio,Support,Resistance,norm_price,bounce_zone,touch_days_ago,avg_15d_at_touch
Date,,,,,,,
2026-06-16,1.389116,1.3,1.4,0.891162,rev res bounce,3.0,1.451574
2026-06-17,1.331337,1.3,1.4,0.313369,rev sup bounce,NaN,NaN
2026-06-18,1.339462,1.3,1.4,0.394620,None,NaN,NaN
2026-06-19,1.324793,1.3,1.4,0.247929,rev sup bounce,NaN,NaN
2026-06-22,1.338458,1.3,1.4,0.384582,None,NaN,NaN
2026-06-23,1.349691,1.3,1.4,0.496905,None,NaN,NaN
2026-06-24,1.371467,1.3,1.4,0.714672,rev res bounce,NaN,NaN
2026-06-25,1.361712,1.3,1.4,0.617117,None,NaN,NaN
2026-06-26,1.362871,1.3,1.4,0.628707,None,NaN,NaN


## **6. FUNCTION TO NORMALIZE 15D AVG PX BEFORE TOUCH**

In [43]:
def add_norm_avg(df):
    df = df.copy()
    df["norm_avg"] = (df["avg_15d_at_touch"] - df["Support"]) / (df["Resistance"] - df["Support"])
    return df

In [44]:
df = add_norm_avg(df)

In [45]:
df[1402-15 : 1405]

,Ratio,Support,Resistance,norm_price,bounce_zone,touch_days_ago,avg_15d_at_touch,norm_avg
Date,,,,,,,,
2026-06-16,1.389116,1.3,1.4,0.891162,rev res bounce,3.0,1.451574,1.515742
2026-06-17,1.331337,1.3,1.4,0.313369,rev sup bounce,NaN,NaN,NaN
2026-06-18,1.339462,1.3,1.4,0.394620,None,NaN,NaN,NaN
2026-06-19,1.324793,1.3,1.4,0.247929,rev sup bounce,NaN,NaN,NaN
2026-06-22,1.338458,1.3,1.4,0.384582,None,NaN,NaN,NaN
2026-06-23,1.349691,1.3,1.4,0.496905,None,NaN,NaN,NaN
2026-06-24,1.371467,1.3,1.4,0.714672,rev res bounce,NaN,NaN,NaN
2026-06-25,1.361712,1.3,1.4,0.617117,None,NaN,NaN,NaN
2026-06-26,1.362871,1.3,1.4,0.628707,None,NaN,NaN,NaN


## **7. FUNCTION FOR APPROACH CHECK**

In [46]:
def add_approach_check(df):
    df = df.copy()
    
    approach = []
    
    for i in range(len(df)):
        zone = df["bounce_zone"].iloc[i]
        norm_avg = df["norm_avg"].iloc[i]
        
        if pd.isna(norm_avg):
            approach.append(None)
        elif zone == "rev res bounce" and 0 < norm_avg < 0.8:
            approach.append("uptrend approach")
        elif zone == "rev sup bounce" and 0.2 < norm_avg < 1:
            approach.append("downtrend approach")
        else:
            approach.append(None)
    
    df["approach"] = approach
    return df

In [47]:
df = add_approach_check(df)

In [48]:
df[df["touch_days_ago"]>=1].tail()

,Ratio,Support,Resistance,norm_price,bounce_zone,touch_days_ago,avg_15d_at_touch,norm_avg,approach
Date,,,,,,,,,
2026-06-15,1.384101,1.3,1.4,0.841012,rev res bounce,2.0,1.451574,1.515742,None
2026-06-16,1.389116,1.3,1.4,0.891162,rev res bounce,3.0,1.451574,1.515742,None
2026-07-01,1.329071,1.3,1.4,0.290706,rev sup bounce,1.0,1.370785,0.707851,downtrend approach
2026-07-06,1.324083,1.3,1.4,0.240833,rev sup bounce,4.0,1.370785,0.707851,downtrend approach
2026-07-07,1.321296,1.3,1.4,0.212965,rev sup bounce,5.0,1.370785,0.707851,downtrend approach


## **8. FUNCTION FOR SIGNAL V1**

In [49]:
def add_signal_v1(df):
    df = df.copy()
    
    signal = []
    
    for i in range(len(df)):
        zone = df["bounce_zone"].iloc[i]
        approach = df["approach"].iloc[i]
        touch = df["touch_days_ago"].iloc[i]
        
        has_touch = not pd.isna(touch)
        
        if zone == "rev res bounce" and approach == "uptrend approach" and has_touch:
            signal.append("reverting from resistance")
        elif zone == "rev sup bounce" and approach == "downtrend approach" and has_touch:
            signal.append("reverting from support")
        else:
            signal.append(None)
    
    df["signal_v1"] = signal
    return df

In [50]:
df = add_signal_v1(df)

In [51]:
df[~df["signal_v1"].isna()].tail()

,Ratio,Support,Resistance,norm_price,bounce_zone,touch_days_ago,avg_15d_at_touch,norm_avg,approach,signal_v1
Date,,,,,,,,,,
2026-05-06,1.533715,1.5,1.6,0.337154,rev sup bounce,4.0,1.533266,0.332660,downtrend approach,reverting from support
2026-05-11,1.521868,1.5,1.6,0.218683,rev sup bounce,5.0,1.526008,0.260077,downtrend approach,reverting from support
2026-07-01,1.329071,1.3,1.4,0.290706,rev sup bounce,1.0,1.370785,0.707851,downtrend approach,reverting from support
2026-07-06,1.324083,1.3,1.4,0.240833,rev sup bounce,4.0,1.370785,0.707851,downtrend approach,reverting from support
2026-07-07,1.321296,1.3,1.4,0.212965,rev sup bounce,5.0,1.370785,0.707851,downtrend approach,reverting from support


## **9. FUNCTION FOR BLOCK1: NO PRIOR BREACH IN TOUCH ZONE (5D)**

In [52]:
def add_block1_check(df):
    df = df.copy()
    
    block1_pass = []
    
    for i in range(len(df)):
        signal = df["signal_v1"].iloc[i]
        resistance = df["Resistance"].iloc[i]
        support = df["Support"].iloc[i]
        
        if signal is None:
            block1_pass.append(None)
            continue
        
        if i < 5:
            block1_pass.append(None)
            continue
        
        window = df["Ratio"].iloc[i-5:i]  # last 5 days, excluding live day
        
        if signal == "reverting from resistance":
            passed = all(window < 1.03 * resistance)                # if even one day breaks it, all() returns False
        elif signal == "reverting from support":
            passed = all(window > 0.97 * support)                   # if even one day breaks it, all() returns False
        else:
            passed = None
        
        block1_pass.append(passed)
    
    df["block1_pass"] = block1_pass
    return df

In [53]:
df = add_block1_check(df)

## **10. FUNCTION FOR BLOCK2: YESTERDAY IN ZONE CHECK**

In [54]:
def add_block2_check(df):
    df = df.copy()
    
    block2_pass = []
    
    for i in range(len(df)):
        signal = df["signal_v1"].iloc[i]
        resistance = df["Resistance"].iloc[i]
        support = df["Support"].iloc[i]
        
        if signal is None:
            block2_pass.append(None)
            continue
        
        if i < 1:
            block2_pass.append(None)
            continue
        
        yesterday_price = df["Ratio"].iloc[i-1]
        y_norm = (yesterday_price - support) / (resistance - support)
        
        if signal == "reverting from resistance":
            passed = 0.65 <= y_norm <= 0.90
        elif signal == "reverting from support":
            passed = 0.10 <= y_norm <= 0.35
        else:
            passed = None
        
        block2_pass.append(passed)
    
    df["block2_pass"] = block2_pass
    return df

In [55]:
df = add_block2_check(df)

## **FINAL SIGNAL**

In [56]:
def add_final_signal(df):
    df = df.copy()
    
    final_signal = []
    
    for i in range(len(df)):
        signal = df["signal_v1"].iloc[i]
        b1 = df["block1_pass"].iloc[i]
        b2 = df["block2_pass"].iloc[i]
        
        if signal is not None and b1 == True and b2 == True:
            final_signal.append(signal)
        else:
            final_signal.append(None)
    
    df["final_signal"] = final_signal
    return df

In [57]:
df = add_final_signal(df)

In [58]:
df[~df["final_signal"].isna()].tail()

,Ratio,Support,Resistance,norm_price,bounce_zone,touch_days_ago,avg_15d_at_touch,norm_avg,approach,signal_v1,block1_pass,block2_pass,final_signal
Date,,,,,,,,,,,,,
2026-01-23,1.520740,1.5,1.6,0.207400,rev sup bounce,4.0,1.550396,0.503963,downtrend approach,reverting from support,True,True,reverting from support
2026-01-26,1.528271,1.5,1.6,0.282714,rev sup bounce,5.0,1.550396,0.503963,downtrend approach,reverting from support,True,True,reverting from support
2026-04-28,1.530350,1.5,1.6,0.303500,rev sup bounce,2.0,1.533594,0.335940,downtrend approach,reverting from support,True,True,reverting from support
2026-05-06,1.533715,1.5,1.6,0.337154,rev sup bounce,4.0,1.533266,0.332660,downtrend approach,reverting from support,True,True,reverting from support
2026-07-07,1.321296,1.3,1.4,0.212965,rev sup bounce,5.0,1.370785,0.707851,downtrend approach,reverting from support,True,True,reverting from support


In [59]:
#df.index

In [60]:
# Save
df.to_csv("reversion_signals.csv", index=True)

# Read back later
df = pd.read_csv("reversion_signals.csv", index_col=0, parse_dates=True)

# **PROBABILITY ANALYSIS**

In [61]:
df = pd.read_csv("reversion_signals.csv", index_col=0, parse_dates=True)

In [62]:
df.tail()

,Ratio,Support,Resistance,norm_price,bounce_zone,touch_days_ago,avg_15d_at_touch,norm_avg,approach,signal_v1,block1_pass,block2_pass,final_signal
Date,,,,,,,,,,,,,
2026-07-31,1.268992,1.2,1.3,0.689921,rev res bounce,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-03,1.252555,1.2,1.3,0.525548,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-04,1.243725,1.2,1.3,0.437254,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-05,1.250529,1.2,1.3,0.505290,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-06,1.252048,1.2,1.3,0.520481,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [63]:
probability_df = df[["Ratio","Support","Resistance","final_signal"]]
probability_df.tail()

,Ratio,Support,Resistance,final_signal
Date,,,,
2026-07-31,1.268992,1.2,1.3,NaN
2026-08-03,1.252555,1.2,1.3,NaN
2026-08-04,1.243725,1.2,1.3,NaN
2026-08-05,1.250529,1.2,1.3,NaN
2026-08-06,1.252048,1.2,1.3,NaN


In [64]:
probability_df[probability_df["final_signal"] == "reverting from support"].shape

(29, 4)

In [65]:
probability_df[probability_df["final_signal"] == "reverting from resistance"].shape

(20, 4)

In [66]:
results = []

for i in range(len(probability_df)):

    row = probability_df.iloc[i]
    if row["final_signal"] != "reverting from support":
        continue

    entry_price = row["Ratio"]
    support = row["Support"]
    resistance = row["Resistance"]

    print(entry_price,support,resistance)

1.42444189174334 1.4 1.5
1.415672686457358 1.4 1.5
1.4348970866906257 1.4 1.5
1.3147951599358694 1.3 1.4
1.33293120273555 1.3 1.4
1.3273576893274186 1.3 1.4
1.31822042661111 1.3 1.4
1.330118756560141 1.3 1.4
1.3295572267282256 1.3 1.4
1.312938112621126 1.3 1.4
1.4122112271652327 1.4 1.5
1.424282657310692 1.4 1.5
1.4221622628467327 1.4 1.5
1.423494596845961 1.4 1.5
1.4163778610379547 1.4 1.5
1.4260687591865184 1.4 1.5
1.534494151014711 1.5 1.6
1.5272783465046005 1.5 1.6
1.532742025917287 1.5 1.6
1.5100839499344487 1.5 1.6
1.5225348497002713 1.5 1.6
1.5255903608317438 1.5 1.6
1.519541365654304 1.5 1.6
1.5127137019503605 1.5 1.6
1.520740020456064 1.5 1.6
1.5282714476247106 1.5 1.6
1.5303499905079934 1.5 1.6
1.533715370675606 1.5 1.6
1.321296499141963 1.3 1.4


In [67]:
results = []

for i in range(len(probability_df)):

    row = probability_df.iloc[i]
    signal = row["final_signal"]

    if signal not in ["reverting from support","reverting from resistance"]:
        continue

    # store entry info
    entry_date = probability_df.index[i]
    entry_price = row["Ratio"]

    if signal == "reverting from support":
        target = row["Resistance"]
        stop = row["Support"]

    if signal == "reverting from resistance":
            target = row["Support"]
            stop = row["Resistance"]

    for j in range(i+1, len(probability_df)):
         

        future_price = probability_df.iloc[j]["Ratio"]
        exit_date = probability_df.index[j]

        outcome = None

        # Support trade
        if signal == "reverting from support":

            if future_price >= target:
                outcome = "Target"
            elif future_price <= stop:
                outcome = "Stop"

        # Resistance trade
        if signal == "reverting from resistance":

            if future_price <= target:
                outcome = "Target"
            elif future_price >= stop:
                outcome = "Stop"


        if outcome is not None:
            results.append({
                "Entry Date": entry_date,
                "Entry Px": entry_price,
                "Signal": signal,
                "Exit Date": exit_date,
                "Exit Px": future_price,
                "Outcome": outcome,
                "Holding Days": j-i
            })

            break



In [68]:
results_df = pd.DataFrame(results)

In [69]:
results_df

,Entry Date,Entry Px,Signal,Exit Date,Exit Px,Outcome,Holding Days
0,2021-05-06,1.424442,reverting from support,2021-06-04,1.508584,Target,20
1,2021-05-07,1.415673,reverting from support,2021-06-04,1.508584,Target,19
2,2021-05-10,1.434897,reverting from support,2021-06-04,1.508584,Target,18
3,2021-11-04,1.314795,reverting from support,2021-12-03,1.298869,Stop,21
4,2021-11-05,1.332931,reverting from support,2021-12-03,1.298869,Stop,20
5,2021-11-08,1.327358,reverting from support,2021-12-03,1.298869,Stop,19
6,2021-11-09,1.318220,reverting from support,2021-12-03,1.298869,Stop,18
7,2021-12-07,1.330119,reverting from support,2021-12-10,1.276665,Stop,3
8,2022-03-02,1.329557,reverting from support,2022-03-07,1.299124,Stop,3
9,2022-03-09,1.312938,reverting from support,2022-03-10,1.288229,Stop,1


Removing repeating signals (consecutive)

In [70]:
results_df["Entry Date"] = pd.to_datetime(results_df["Entry Date"])

results_df["prev_date"] = results_df["Entry Date"].shift(1)

results_df["date_gap"] = (results_df["Entry Date"] - results_df["prev_date"]).dt.days

results_df["is_new_trade"] = results_df["date_gap"].isna() | (results_df["date_gap"] > 3)

results_df_deduped = results_df[results_df["is_new_trade"]]

results_df_deduped

,Entry Date,Entry Px,Signal,Exit Date,Exit Px,Outcome,Holding Days,prev_date,date_gap,is_new_trade
0,2021-05-06,1.424442,reverting from support,2021-06-04,1.508584,Target,20,NaT,NaN,True
3,2021-11-04,1.314795,reverting from support,2021-12-03,1.298869,Stop,21,2021-05-10,178.0,True
7,2021-12-07,1.330119,reverting from support,2021-12-10,1.276665,Stop,3,2021-11-09,28.0,True
8,2022-03-02,1.329557,reverting from support,2022-03-07,1.299124,Stop,3,2021-12-07,85.0,True
9,2022-03-09,1.312938,reverting from support,2022-03-10,1.288229,Stop,1,2022-03-02,7.0,True
10,2022-05-23,1.275743,reverting from resistance,2022-06-02,1.301415,Stop,8,2022-03-09,75.0,True
12,2022-06-01,1.287778,reverting from resistance,2022-06-02,1.301415,Stop,1,2022-05-24,8.0,True
13,2022-07-21,1.469515,reverting from resistance,2022-08-03,1.372451,Target,9,2022-06-01,50.0,True
15,2022-08-30,1.368897,reverting from resistance,2022-09-05,1.421634,Stop,4,2022-07-22,39.0,True
17,2022-09-27,1.385558,reverting from resistance,2022-09-30,1.407400,Stop,3,2022-08-31,27.0,True


In [71]:
df = results_df_deduped.copy()

# 1. sign column: +1 for long (support), -1 for short (resistance)
df["sign"] = df["Signal"].map({"reverting from support": 1, "reverting from resistance": -1})

# 2. returns column
df["return_pct"] = df["sign"] * (df["Exit Px"] - df["Entry Px"]) / df["Entry Px"]

df.tail()

,Entry Date,Entry Px,Signal,Exit Date,Exit Px,Outcome,Holding Days,prev_date,date_gap,is_new_trade,sign,return_pct
38,2025-12-12,1.576920,reverting from resistance,2026-01-13,1.495307,Target,17,2025-10-20,53.0,True,-1,0.051754
42,2026-01-21,1.519541,reverting from support,2026-03-03,1.498002,Stop,29,2025-12-17,35.0,True,1,-0.014175
46,2026-04-28,1.530350,reverting from support,2026-04-30,1.485763,Stop,2,2026-01-26,92.0,True,1,-0.029135
47,2026-05-06,1.533715,reverting from support,2026-05-14,1.497060,Stop,6,2026-04-28,8.0,True,1,-0.023900
48,2026-07-07,1.321296,reverting from support,2026-07-14,1.270672,Stop,5,2026-05-06,62.0,True,1,-0.038315


In [72]:
#print(df.shape)
print(f"Total unique signals: {df.shape[0]}")

Total unique signals: 25


In [73]:
df[df["Outcome"] == "Target"].shape[0] / df.shape[0]

winrate = df[df["Outcome"] == "Target"].shape[0] / df.shape[0]

print(f"Win rate: {winrate*100:.0f}%")

Win rate: 20%


In [74]:
df.loc[df['Outcome'] == "Target", "return_pct"]
df.loc[df['Outcome'] == "Target", "return_pct"].mean()

avgwin = df.loc[df['Outcome'] == "Target", "return_pct"].mean()

print(f"Avg win%: {avgwin*100:.2f}%")

Avg win%: 5.75%


In [75]:
df.loc[df['Outcome'] == "Stop", "return_pct"]
df.loc[df['Outcome'] == "Stop", "return_pct"].mean()

avgloss = df.loc[df['Outcome'] == "Stop", "return_pct"].mean()

print(f"Avg loss%: {avgloss*100:.2f}%")

Avg loss%: -2.37%


In [76]:
ev = winrate*avgwin + (1-winrate)*avgloss
print(f"Expected value per trade: {ev*100:.2f}%")

Expected value per trade: -0.75%
